In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

In [2]:
def load_data(file_path: str):
    """
    Loads the safrin03 dataset
    """
    # 1. Load the data defensively
    df = pd.read_csv(file_path)
    return df


In [4]:
def preprocess_data(df: pd.DataFrame):
    """
    Prepares the feature preprocessing pipeline.
    """
    # 1. Load the data defensively
    df = pd.read_csv(file_path)
    # Clean column names to lowercase/underscores to avoid syntax problems in SQL or cloud logs
    df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')

    # 2. Separate target from features
    # (Assuming the target column name becomes 'churn' after formatting)
    X = df.drop(columns=['churn', 'customer_id'], errors='ignore')
    y = df['churn']

    # 3. Identify feature types automatically or explicitly
    numeric_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
    categorical_features = X.select_dtypes(include=['object', 'category']).columns.tolist()

    # 4. Define transformations per data type (Best Practice)
    # Numerical: Impute missing values with median, then scale
    numeric_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ])

    # Categorical: Impute missing values with most frequent string, then One-Hot Encode
    categorical_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ])

    # 5. Bundle transformers into a ColumnTransformer
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', numeric_transformer, numeric_features),
            ('cat', categorical_transformer, categorical_features)
        ],
        remainder='drop' # Explicitly drop columns we didn't specify (like IDs)
    )

    # 6. Train/Test Split BEFORE fitting anything to avoid data leakage
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.20, stratify=y, random_state=42
    )

    return X_train, X_test, y_train, y_test, preprocessor

In [ ]:
if __name__ == "__main__":
    # Example execution within your Docker container volume path
    data_descriptions_df = load_data('/Users/annamorgiel/code/soodabeh/retention-agent/raw_data/data_descriptions.csv')
    data_train_df = load_data('/Users/annamorgiel/code/soodabeh/retention-agent/raw_data/train.csv')
    data_test_df = load_data('/Users/annamorgiel/code/soodabeh/retention-agent/raw_data/test.csv')
    #X_train, X_test, y_train, y_test, preprocessor  = preprocess_data(data_df)
    print(f"data_descriptions_df shape: {data_descriptions_df.shape}")
    print(f"data_train_df shape: {data_train_df.shape}")

    #TODO we only need train dataset, split it into train test val
    print(f"data_test_df shape: {data_test_df.shape}")

data_descriptions_df shape: (21, 4)
data_train_df shape: (243787, 21)
data_test_df shape: (104480, 20)


In [13]:
data_descriptions_df

,Column_name,Column_type,Data_type,Description
0,AccountAge,Feature,integer,The age of the user's account in months.
1,MonthlyCharges,Feature,float,The amount charged to the user on a monthly ba...
2,TotalCharges,Feature,float,The total charges incurred by the user over th...
3,SubscriptionType,Feature,object,The type of subscription chosen by the user (B...
4,PaymentMethod,Feature,string,The method of payment used by the user.
5,PaperlessBilling,Feature,string,Indicates whether the user has opted for paper...
6,ContentType,Feature,string,The type of content preferred by the user (Mov...
7,MultiDeviceAccess,Feature,string,Indicates whether the user has access to the s...
8,DeviceRegistered,Feature,string,"The type of device registered by the user (TV,..."
9,ViewingHoursPerWeek,Feature,float,The number of hours the user spends watching c...


In [11]:
data_train_df.head(4)

,AccountAge,MonthlyCharges,TotalCharges,SubscriptionType,PaymentMethod,PaperlessBilling,ContentType,MultiDeviceAccess,DeviceRegistered,ViewingHoursPerWeek,...,ContentDownloadsPerMonth,GenrePreference,UserRating,SupportTicketsPerMonth,Gender,WatchlistSize,ParentalControl,SubtitlesEnabled,CustomerID,Churn
0,20,11.055215,221.104302,Premium,Mailed check,No,Both,No,Mobile,36.758104,...,10,Sci-Fi,2.176498,4,Male,3,No,No,CB6SXPNVZA,0
1,57,5.175208,294.986882,Basic,Credit card,Yes,Movies,No,Tablet,32.450568,...,18,Action,3.478632,8,Male,23,No,Yes,S7R2G87O09,0
2,73,12.106657,883.785952,Basic,Mailed check,Yes,Movies,No,Computer,7.395160,...,23,Fantasy,4.238824,6,Male,1,Yes,Yes,EASDC20BDT,0
3,32,7.263743,232.439774,Basic,Electronic check,No,TV Shows,No,Tablet,27.960389,...,30,Drama,4.276013,2,Male,24,Yes,Yes,NPF69NT69N,0


In [12]:
data_test_df.head(4)

,AccountAge,MonthlyCharges,TotalCharges,SubscriptionType,PaymentMethod,PaperlessBilling,ContentType,MultiDeviceAccess,DeviceRegistered,ViewingHoursPerWeek,AverageViewingDuration,ContentDownloadsPerMonth,GenrePreference,UserRating,SupportTicketsPerMonth,Gender,WatchlistSize,ParentalControl,SubtitlesEnabled,CustomerID
0,38,17.869374,679.036195,Premium,Mailed check,No,TV Shows,No,TV,29.126308,122.274031,42,Comedy,3.522724,2,Male,23,No,No,O1W6BHP6RM
1,77,9.912854,763.289768,Basic,Electronic check,Yes,TV Shows,No,TV,36.873729,57.093319,43,Action,2.021545,2,Female,22,Yes,No,LFR4X92X8H
2,5,15.019011,75.095057,Standard,Bank transfer,No,TV Shows,Yes,Computer,7.601729,140.414001,14,Sci-Fi,4.806126,2,Female,22,No,Yes,QM5GBIYODA
3,88,15.357406,1351.451692,Standard,Electronic check,No,Both,Yes,Tablet,35.586430,177.002419,14,Comedy,4.943900,0,Female,23,Yes,Yes,D9RXTK2K9F


In [16]:
data_train_df.columns

Index(['AccountAge', 'MonthlyCharges', 'TotalCharges', 'SubscriptionType',
       'PaymentMethod', 'PaperlessBilling', 'ContentType', 'MultiDeviceAccess',
       'DeviceRegistered', 'ViewingHoursPerWeek', 'AverageViewingDuration',
       'ContentDownloadsPerMonth', 'GenrePreference', 'UserRating',
       'SupportTicketsPerMonth', 'Gender', 'WatchlistSize', 'ParentalControl',
       'SubtitlesEnabled', 'CustomerID', 'Churn'],
      dtype='object')

In [19]:
y = data_train_df['Churn']
X = data_train_df.drop(columns=['Churn'])


print(f"y shape: {y.shape}")
print(f"X shape: {X.shape}")

y shape: (243787,)
X shape: (243787, 20)


In [20]:
X.value_counts()

AccountAge  MonthlyCharges  TotalCharges  SubscriptionType  PaymentMethod     PaperlessBilling  ContentType  MultiDeviceAccess  DeviceRegistered  ViewingHoursPerWeek  AverageViewingDuration  ContentDownloadsPerMonth  GenrePreference  UserRating  SupportTicketsPerMonth  Gender  WatchlistSize  ParentalControl  SubtitlesEnabled  CustomerID
1           4.991154        4.991154      Premium           Credit card       No                Both         Yes                Mobile            1.559813             72.489247               20                        Drama            2.860004    7                       Male    4              No               Yes               3FNAWWWQOA    1
80          12.256023       980.481869    Standard          Bank transfer     Yes               Movies       Yes                Computer          27.174994            86.773273               28                        Action           2.472908    7                       Female  5              No               No     

In [21]:
X.describe()

,AccountAge,MonthlyCharges,TotalCharges,ViewingHoursPerWeek,AverageViewingDuration,ContentDownloadsPerMonth,UserRating,SupportTicketsPerMonth,WatchlistSize
count,243787.000000,243787.000000,243787.000000,243787.000000,243787.000000,243787.000000,243787.000000,243787.000000,243787.000000
mean,60.083758,12.490695,750.741017,20.502179,92.264061,24.503513,3.002713,4.504186,12.018508
std,34.285143,4.327615,523.073273,11.243753,50.505243,14.421174,1.155259,2.872548,7.193034
min,1.000000,4.990062,4.991154,1.000065,5.000547,0.000000,1.000007,0.000000,0.000000
25%,30.000000,8.738543,329.147027,10.763953,48.382395,12.000000,2.000853,2.000000,6.000000
50%,60.000000,12.495555,649.878487,20.523116,92.249992,24.000000,3.002261,4.000000,12.000000
75%,90.000000,16.238160,1089.317362,30.219396,135.908048,37.000000,4.002157,7.000000,18.000000
max,119.000000,19.989957,2378.723844,39.999723,179.999275,49.000000,4.999989,9.000000,24.000000
